# OTel Metrics ETL - Interactive Example

This notebook demonstrates the OTel Metrics ETL pipeline for transforming high-cardinality OpenTelemetry metrics into ML-ready features.

In [ ]:
import sys
sys.path.insert(0, '..')

from datetime import datetime, timedelta
import pandas as pd
import numpy as np

from otel_etl import run_profiler, fetch_and_denormalize
from signals import PrometheusClient
from otel_etl.feature_generator.schema_registry import SchemaRegistry

# Configuration
PROMETHEUS_URL = "http://10.0.0.5:9090"

pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 120)

## Step 1: Profile Metrics

Run the profiler to discover metrics, labels, and cardinality.

In [ ]:
schema = run_profiler(
    prometheus_url=PROMETHEUS_URL,
    output_path="schema_config.yaml",
    profiling_window_hours=1,
    cardinality_thresholds={
        "tier1_max": 10,
        "tier2_max": 50,
        "tier3_max": 200,
    },
)

print(f"Discovered {len(schema['metrics'])} metric families")
print(f"Profiled at: {schema['profiled_at']}")

### Inspect Schema Config

In [ ]:
# Show a sample metric family
sample_family = list(schema['metrics'].keys())[0]
print(f"\nSample metric family: {sample_family}")
print(f"Type: {schema['metrics'][sample_family]['type']}")
print(f"\nLabels:")
for label, info in schema['metrics'][sample_family]['labels'].items():
    print(f"  {label}: tier={info['tier']}, cardinality={info['cardinality']}, action={info['action']}")

## Step 2: Fetch and Transform Metrics

In [ ]:
end_time = datetime.utcnow()
start_time = end_time - timedelta(minutes=30)

features_df = fetch_and_denormalize(
    prometheus_url=PROMETHEUS_URL,
    start=start_time,
    end=end_time,
    step="60s",
    schema_config="schema_config.yaml",
    column_registry="column_registry.yaml",
    layers=[1, 2, 3],
    include_deltas=True,
)

print(f"Shape: {features_df.shape}")
print(f"Entities: {features_df['entity_key'].nunique()}")
print(f"Features: {len([c for c in features_df.columns if c not in ['timestamp', 'entity_key']])}")

### Inspect Output

In [ ]:
# Show first few rows
features_df.head()

In [ ]:
# Show feature columns
feature_cols = [c for c in features_df.columns if c not in ['timestamp', 'entity_key']]
print(f"Total features: {len(feature_cols)}")
print(f"\nSample features:")
for col in feature_cols[:10]:
    print(f"  - {col}")

## Step 3: Analyze Features

In [ ]:
# Completeness analysis
completeness_by_column = features_df[feature_cols].notna().mean()
print(f"Median completeness: {completeness_by_column.median():.1%}")
print(f"Sparse columns (>50% NaN): {(completeness_by_column < 0.5).sum()}")

# Show sparsest columns
print("\nSparsest features:")
print(completeness_by_column.nsmallest(5))

In [ ]:
# Feature value distribution
import matplotlib.pyplot as plt

# Plot histogram of a sample feature
sample_feature = feature_cols[0]
features_df[sample_feature].hist(bins=30, figsize=(10, 4))
plt.title(f"Distribution: {sample_feature}")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()

## Step 4: Feature Engineering Examples

In [ ]:
# Filter to a specific entity
entities = features_df['entity_key'].unique()
if len(entities) > 0:
    entity = entities[0]
    entity_df = features_df[features_df['entity_key'] == entity].copy()
    
    print(f"Entity: {entity}")
    print(f"Rows: {len(entity_df)}")
    
    # Plot time series of a feature
    if len(feature_cols) > 0:
        feature = feature_cols[0]
        entity_df.plot(x='timestamp', y=feature, figsize=(12, 4), title=f"{entity} - {feature}")
        plt.show()

In [ ]:
# Find highly correlated features
corr_matrix = features_df[feature_cols].corr()
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print(f"Found {len(high_corr_pairs)} highly correlated feature pairs (>0.9):")
    for feat1, feat2, corr in high_corr_pairs[:5]:
        print(f"  {corr:.3f}: {feat1} <-> {feat2}")
else:
    print("No highly correlated features found")

## Step 5: Export for ML

In [ ]:
# Prepare for ML: drop index columns, fill NaNs
X = features_df[feature_cols].fillna(0)

print(f"Feature matrix shape: {X.shape}")
print(f"Data type: {X.dtypes[0]}")
print(f"Memory usage: {X.memory_usage().sum() / 1024**2:.1f} MB")

# Save
X.to_csv("otel_tel_features.csv", index=False)
print("\n✓ Saved otel_tel_features.csv")